# tiled_vs_manual：`tiled_copy` / `tiled_mma` 和手算偏移

[GEMM 教程](../kernels/gemm/README.md) 的 v0/v1/v2 都是"手算"写法：用 `make_ordered_layout` 把矩阵拆成若干维，自己算出每个 lane 的行号、K 分组（`row = lane % 16`、`kg = lane // 16`），`.load()` 读进寄存器，再按 MFMA 的操作数排布调用 `rocdl.mfma_*`。

FlyDSL 还有一套 CuTe 风格的写法：

- `tiled_mma`：MFMA atom + 怎样铺开（几个 wave、K 怎样置换），它知道 A/B/C 的每个元素在哪个 lane 的哪个寄存器里；
- `tiled_copy`：copy atom（一次搬多少位）+ 线程排布；`make_tiled_copy_A/B/C(atom, tiled_mma)` 直接从 `tiled_mma` 推出和 MFMA 操作数一致的搬运方式；
- `partition_S/D`、`make_fragment_A/B/C`、`retile`：按上面两者把 global / LDS tensor 切成每个线程自己的那一份；
- `fx.copy(atom, src, dst)`、`fx.gemm(atom, d, a, b, c)`：对切好的 tensor 整体执行。

本实验先用几张图和两个最小例子介绍这套写法，再用同一个 bf16 GEMM（`C[M,N] = A[M,K] · B[N,K]^T`，fp32 输出）比较两种写法：写法与代码量、生成的 ISA、编译时间、性能（并深入分析地址计算和 MFMA 遍历顺序）、换 MFMA 指令、LDS + swizzle、调度控制，最后给出何时用哪种的建议，并讨论这套 API 不直观、学习曲线陡的问题（第 8 节）。

ISA 同时编译 gfx942 / gfx950，正确性和性能在本机 gfx950 上测（v0/v1/v2 的 tile 配置是按 gfx942 选的，绝对 TFLOPS 只作相对比较）。代码里 `# →` 开头的注释是实测结果，耗时类数字每次运行会有几个百分点的波动。

## 0. 准备

- `isa_stats(build, *args)`：分别编译 gfx942 / gfx950（和 GEMM 教程的 `common.py` 一样，编译选项 `opt_level=2`、`amdgpu-mfma-vgpr-form=False`），统计主循环的指令、VGPR 等；`dumps(...)` 返回编译各阶段的文件（LLVM IR、ISA）；
- `measure(build, A, B, C)`：在本机编译一次（记下编译时间）、检查结果、测 kernel 时间（20 次取中位数）；
- `lines(fn)`：函数源码去掉空行和注释后的行数。

In [1]:
import os, sys
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"
sys.path.insert(0, os.path.abspath("../kernels/gemm"))

import contextlib, csv, glob, inspect, io, re, shutil, subprocess, tempfile, time, unicodedata
from collections import Counter
from importlib.metadata import version

import torch
import flydsl.compiler as flyc
import flydsl.expr as fx
from flydsl.expr import gpu
from flydsl._mlir import ir

from v0 import build_v0
from v1 import build_v1
from v2 import build_v2

ARCHS = ("gfx942", "gfx950")
LOCAL_ARCH = torch.cuda.get_device_properties(0).gcnArchName.split(":")[0]
HINTS = {"opt_level": 2, "llvm_options": {"amdgpu-mfma-vgpr-form": False}}
stream = fx.Stream(torch.cuda.current_stream().cuda_stream)
print("flydsl", version("flydsl"), "| torch", torch.__version__, "| 本机 GPU", LOCAL_ARCH)


def dumps(launcher, *args, arch):
    d = tempfile.mkdtemp(prefix="tiled_")
    env = dict(ARCH=arch, COMPILE_ONLY="1", FLYDSL_DUMP_IR="1", FLYDSL_DUMP_DIR=d)
    old = {k: os.environ.get(k) for k in env}
    os.environ.update(env)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            flyc.compile[HINTS](launcher, *args)
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k)
            else:
                os.environ[k] = v
    return {os.path.basename(p): open(p).read() for p in glob.glob(f"{d}/**/*.*", recursive=True)}


def isa(launcher, *args, arch):
    return dumps(launcher, *args, arch=arch)["22_final_isa.s"]


def body(text):
    lines_, inside = [], False
    for raw in text.splitlines():
        if re.match(r"^\w+:", raw):
            inside = True
            continue
        line = raw.split(";")[0].strip()
        if inside and line and (not line.startswith(".") or line.startswith(".LBB")):
            lines_.append(line)
        if line.startswith("s_endpgm"):
            break
    return lines_


def main_loop(lines_):
    for i, l in enumerate(lines_):
        if l.endswith(":"):
            for j in range(i + 1, len(lines_)):
                if lines_[j].startswith("s_cbranch") and lines_[j].endswith(l[:-1]):
                    return lines_[i:j + 1]
    return []


def stats(text):
    b = body(text)
    loop = main_loop(b)
    ops = Counter(l.split()[0] for l in loop)
    meta = lambda k: int(re.search(rf"\.{k}:\s+(\d+)", text).group(1))
    return {
        "总指令": len(b), "循环指令": len(loop),
        "MFMA": sum(v for k, v in ops.items() if "mfma" in k),
        "global_load": sum(v for k, v in ops.items() if k.startswith("global_load")),
        "ds_read/write": sum(v for k, v in ops.items() if k.startswith("ds_")),
        "循环 VALU": sum(v for k, v in ops.items() if k.startswith("v_") and "mfma" not in k and "accvgpr" not in k),
        "accvgpr 搬运": sum(v for k, v in ops.items() if "accvgpr" in k),
        "VGPR": meta("vgpr_count"),         # gfx90a 之后是 VGPR + AGPR 的总数，最多 512
    }


def isa_stats(build, *args):
    return {arch: stats(isa(build(), *args, arch=arch)) for arch in ARCHS}


def measure(build, A, B, C, ref, iters=20):
    C.zero_()
    t0 = time.perf_counter()
    f = flyc.compile[HINTS](build(), A, B, C, stream)      # 编译并运行一次
    torch.cuda.synchronize()
    compile_ms = (time.perf_counter() - t0) * 1e3
    ok = torch.allclose(C, ref, atol=1e-3, rtol=1e-3)
    ts = []
    for _ in range(iters):
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record()
        f(A, B, C, stream)
        e.record()
        torch.cuda.synchronize()
        ts.append(s.elapsed_time(e) * 1e3)
    us = sorted(ts)[iters // 2]
    M, K = A.shape
    return {"正确": ok, "编译 ms": round(compile_ms), "us": round(us, 1),
            "TFLOPS": round(2 * M * B.shape[0] * K / us / 1e6, 1)}


def lines(fn):
    src = inspect.getsource(fn).splitlines()
    return sum(1 for l in src if l.strip() and not l.strip().startswith(("#", '"""')))


def width(s):
    return sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in str(s))


def table(rows):
    """rows: {名字: {列: 值}}"""
    cols = list(next(iter(rows.values())))
    first = max(width(n) for n in rows)
    ws = [max(width(c), *(width(r[c]) for r in rows.values())) for c in cols]
    pad = lambda s, w: " " * (w - width(s)) + str(s)
    print(" " * first + " | " + " | ".join(pad(c, w) for c, w in zip(cols, ws)))
    for n, r in rows.items():
        print(str(n) + " " * (first - width(n)) + " | " + " | ".join(pad(r[c], w) for c, w in zip(cols, ws)))

flydsl 0.3.4.1 | torch 2.9.1+rocm7.2.0.git7e1940d4 | 本机 GPU gfx950


## 1. 入门：tiled_copy 和 tiled_mma 怎么用

本节的图都是用 FlyDSL 在 host 上算出的 layout 直接画的。图下方打印的 layout 串怎么读，见 1.5 节。

### 1.1 atom：一条指令做什么

**MMA atom** 描述一条 MFMA 指令：`fx.make_mma_atom(fx.rocdl.MFMA(16, 16, 16, fx.BFloat16))` 是一个 wave 算 `C[16x16] += A[16x16] · B[16x16]^T`。它自带三个 **TV layout**（thread-value）：第 `t` 个 lane 的第 `v` 个值是矩阵里的哪个元素。下图是 A 和 C 的 TV layout（B 和 A 一样，把 M 换成 N）：

![MFMA 16x16x16 的 A / C 排布](figs/tiled_atom.svg)

- A：lane `l` 拿第 `l % 16` 行、第 `4*(l//16)` 开始的 4 个连续 K（64 位寄存器）。
- C：lane `l` 拿第 `l % 16` 列、第 `4*(l//16)` 开始的 4 行（4 个 fp32 寄存器）。

这就是 v0 里手写的 `row = lane % 16`、`kg = lane // 16`。

**copy atom** 描述一条访存指令一次搬多少：`fx.make_copy_atom(fx.UniversalCopy128b(), fx.BFloat16)` 是每个线程一次搬 128 位（8 个 bf16），按源 / 目的地址空间变成 `global_load_dwordx4`、`ds_write_b128` 等。还有 `BufferCopy128b`（buffer load）、`BufferCopyLDS`（DMA 到 LDS）等，见 [LDS_101](lds_101.ipynb)。

### 1.2 tiled_mma：atom 怎样铺开

`fx.make_tiled_mma(atom, atom_layout, permutation)`：

- `atom_layout`：M / N / K 方向各摆几个 atom，每个 atom 由一个 wave 执行。`(1, 1, 1)` 是 1 个 wave；`(2, 2, 1):(1, 2, 0)` 是 4 个 wave 排成 2x2；
- `permutation`：可选，对 M / N / K 坐标重新排列，改变元素分到哪个 lane。下面的 `(4,4,2):(1,8,4)` 让每个 lane 拿 8 个连续 K，分给两条 MFMA——就是 v0 的做法；
- 一个 tiled_mma 覆盖的区域叫 tile（`tile_size_mnk`），更大的矩阵就是重复这个 tile。

![tiled_mma 的 K 置换和 2x2 wave](figs/tiled_mma.svg)

左：同样是 16x32 的 A，不置换时 lane 组在每条 MFMA 里只拿 4 个连续 K，要两次 64 位 load；置换后每个 lane 组拿 8 个连续 K，一次 128 位 load，前 4 个给第 0 条 MFMA、后 4 个给第 1 条。A 和 B 用同一个置换，点积的求和顺序变了，结果不变。右：`(2, 2, 1)` 的 tile 是 32x32，每个 wave 算其中 16x16；更大的块是这个 32x32 的重复，所以一个 wave 的数据每隔 16 行出现一次（v1 的手写版是每个 wave 一整块连续的 128 行，分法不同但都对）。

### 1.3 tiled_copy：一组线程怎样搬一块数据

两种构造方式：

- `fx.make_tiled_copy_tv(atom, thr_layout, val_layout)`：自己指定线程怎样排、每个线程拿哪几个值。常用于 global → LDS 这类只要合并访存、和 MFMA 排布无关的搬运。下图是 256 个线程搬 32x64 的 bf16：每行 8 个线程、每个 16 B；
- `fx.make_tiled_copy_A/B/C(atom, tiled_mma)`：线程和值的排布直接取自 tiled_mma 的 A / B / C，搬到寄存器里的数据正好是 MFMA 要的样子。

![make_tiled_copy_tv 的线程排布](figs/tiled_copy.svg)

### 1.4 从"整块"到"每个线程的一份"

tiled_copy / tiled_mma 描述的是整个线程组。写 kernel 时先用当前线程号切出自己的视角，再把 tensor 切成自己的那一份：

| API | 作用 | 结果的 shape |
|---|---|---|
| `tiled_copy.get_slice(tid)` / `tiled_mma.thr_slice(tid)` | 取第 `tid` 个线程的视角 | |
| `thr_copy.partition_S(src)` / `partition_D(dst)` | 源 / 目的 tensor 里属于本线程的元素 | `(V, 重复 M, 重复 N, 其余维...)`：`V` 是一次 copy atom 的值，后面是 tile 在各维重复的次数 |
| `thr_mma.make_fragment_A/B/C(t)` | 按 MFMA 操作数的排布分配寄存器 tensor | `(V, 重复 M, 重复 K)`，`V` 是一条 MFMA 的操作数 |
| `fx.make_fragment_like(t)` | 分配和 `t` 同形状的寄存器 tensor | 同 `t` |
| `thr_copy.retile(frag)` | 把 MFMA 视角的寄存器 tensor 换成 copy 视角（同一块寄存器） | copy 的 `(V, ...)` |
| `fx.copy(atom, src, dst)` | 对重复的维度循环，每个位置调用一次 copy atom | |
| `fx.gemm(atom, D, A, B, C)` | 对 M / N / K 的重复循环，每个位置调用一次 MFMA；顺序由 `traversal_order` 决定（4.2 节） | |

它们之间的关系（以 A 为例）：

![tiled API 的数据流](figs/tiled_flow.svg)

两个最小例子。第一个只用 tiled_copy：256 个线程把 64x64 的 bf16 经过寄存器拷到另一个 tensor（trace 时打印各 tensor 的类型）：

In [2]:
@flyc.kernel(known_block_size=(256, 1, 1))
def tile_copy(X: fx.Tensor, Y: fx.Tensor):
    tid = fx.thread_idx.x
    gX = fx.make_view(fx.get_iter(X), fx.make_layout((64, 64), (64, 1)))     # 行主序 64x64
    gY = fx.make_view(fx.get_iter(Y), fx.make_layout((64, 64), (64, 1)))
    cp = fx.make_copy_atom(fx.UniversalCopy128b(), fx.BFloat16)
    tiled_copy = fx.make_tiled_copy_tv(cp, fx.make_layout((32, 8), (8, 1)), fx.make_layout((1, 8), (1, 1)))
    thr_copy = tiled_copy.get_slice(tid)
    src = thr_copy.partition_S(gX)
    dst = thr_copy.partition_D(gY)
    frag = fx.make_fragment_like(src)
    print("tile_mn  :", tiled_copy.tile_mn)           # → [32|64]：一次搬 32 行 x 64 列
    print("layout_tv:", tiled_copy.layout_tv_tiled)   # → ((8,32),8):((256,1),32)：线程 (列块, 行)，8 个值沿列
    print("src      :", src.type)                     # → ((8,1),2,1)：每次 8 个值，行方向重复 2 次（64/32）
    print("frag     :", frag.type)                    # → 同样的 shape，在寄存器里
    fx.copy(cp, src, frag)                            # → 2 条 global_load_dwordx4
    fx.copy(cp, frag, dst)                            # → 2 条 global_store_dwordx4


@flyc.jit
def run_copy(X: fx.Tensor, Y: fx.Tensor, stream: fx.Stream):
    tile_copy(X, Y).launch(grid=(1, 1, 1), block=(256, 1, 1), stream=stream)


X = torch.randn(64, 64, dtype=torch.bfloat16, device="cuda")
Y = torch.zeros_like(X)
run_copy(X, Y, stream)
torch.cuda.synchronize()
print("本机 GPU 正确:", torch.equal(X, Y))           # → True

tile_mn  : Tile<[32|64]>
layout_tv: Layout<((8,32),8):((256,1),32)>
src      : !fly.memref<bf16, global, ((8,1),2,1):((1,0),2048,0)>
frag     : !fly.memref<bf16, register, ((8,1),2,1):((1,0),8,0)>
本机 GPU 正确: True


第二个用 tiled_mma：1 个 wave 算 32x32x32 的 `C = A · B^T`，A / B 直接从 global 读进 MFMA 的寄存器：

In [3]:
@flyc.kernel(known_block_size=(64, 1, 1))
def tiny_gemm(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor):
    tid = fx.thread_idx.x
    gA = fx.make_view(fx.get_iter(A), fx.make_layout((32, 32), (32, 1)))     # (M, K)，K 连续
    gB = fx.make_view(fx.get_iter(B), fx.make_layout((32, 32), (32, 1)))     # (N, K)
    gC = fx.make_view(fx.get_iter(C), fx.make_layout((32, 32), (32, 1)))     # (M, N)
    atom = fx.make_mma_atom(fx.rocdl.MFMA(16, 16, 16, fx.BFloat16))
    tiled_mma = fx.make_tiled_mma(atom, fx.make_layout((1, 1, 1), (0, 0, 0)),
                                  fx.make_tile(None, None, fx.make_layout((4, 4, 2), (1, 8, 4))))
    thr_mma = tiled_mma.thr_slice(tid)
    cp = fx.make_copy_atom(fx.UniversalCopy128b(), fx.BFloat16)
    tcA = fx.make_tiled_copy_A(cp, tiled_mma).get_slice(tid)
    tcB = fx.make_tiled_copy_B(cp, tiled_mma).get_slice(tid)
    fA, fB = thr_mma.make_fragment_A(gA), thr_mma.make_fragment_B(gB)
    fC = thr_mma.make_fragment_C(gC)
    print("tile_size_mnk:", tiled_mma.tile_size_mnk)          # → (16,16,32)
    print("tcA.partition_S(gA):", tcA.partition_S(gA).type)  # → ((8,1),2,1)：8 个连续 K，M 方向重复 2 次
    print("fA:", fA.type)                                     # → (4,2,2)：每条 MFMA 4 个值，M 重复 2，K 重复 2
    print("tcA.retile(fA):", tcA.retile(fA).type)            # → ((8,1),2,1)：同一块寄存器，按 copy 的形状
    print("fC:", fC.type)                                     # → ((4,1),2,2)：每条 MFMA 4 个累加值，M、N 各重复 2
    fx.copy(cp, tcA.partition_S(gA), tcA.retile(fA))         # → 2 条 global_load_dwordx4
    fx.copy(cp, tcB.partition_S(gB), tcB.retile(fB))
    fC.fill(0)
    fx.gemm(atom, fC, fA, fB, fC)                             # → 2 x 2 x 2 = 8 条 MFMA
    cpC = fx.make_copy_atom(fx.UniversalCopy32b(), fx.Float32)
    tcC = fx.make_tiled_copy_C(cpC, tiled_mma).get_slice(tid)
    fx.copy(cpC, tcC.retile(fC), tcC.partition_S(gC))


@flyc.jit
def run_gemm(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor, stream: fx.Stream):
    tiny_gemm(A, B, C).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)


A = torch.randn(32, 32, dtype=torch.bfloat16, device="cuda")
B = torch.randn(32, 32, dtype=torch.bfloat16, device="cuda")
C = torch.zeros(32, 32, device="cuda")
run_gemm(A, B, C, stream)
torch.cuda.synchronize()
print("本机 GPU 正确:", torch.allclose(C, A.float() @ B.float().T, atol=1e-2, rtol=1e-2))   # → True

tile_size_mnk: IntTuple<(16,16,32)>
tcA.partition_S(gA): !fly.memref<bf16, global, ((8,1),2,1):((1,0),512,0)>
fA: !fly.memref<bf16, register, (4,2,2):(1,8,4)>
tcA.retile(fA): !fly.memref<bf16, register, ((8,1),2,1):((1,0),8,0)>
fC: !fly.memref<f32, register, ((4,1),2,2):((1,0),8,4)>
本机 GPU 正确: True


整个过程没有一处写 lane 号或偏移：元素分配全部由 atom 和 layout 推出。

### 1.5 如何读 TV layout

上面打印出来的 `((16,4),(4,(1,2))):((1,128),(16,(0,64)))` 之类的串信息是完整的，但不好读。把它翻译成 v0 那样的公式只需要四条规则：

1. **`shape:stride`**：冒号左边是每一维的大小，右边是对应的步长，结构一一对应。坐标对应的偏移 = 各维坐标 × 步长之和。大小为 1 的维只是占位，可以忽略。
2. **TV layout 有两个顶层维度**：第 0 维是线程 `t`，第 1 维是该线程的第 `v` 个值（在寄存器里的位置）。
3. **嵌套的维度按"第一个最快"拆开**：线程维 `(16,4)` 表示 `t = t0 + 16*t1`，即 `t0 = t % 16`、`t1 = t // 16`。
4. **偏移是 tile 里的列主序编号**：对 `TM` 行的 tile，`行 = 偏移 % TM`、`列 = 偏移 // TM`。所以步长小于 `TM` 的维度沿行走，是 `TM` 倍数的维度沿列走（每步 `步长 / TM` 列）。

以 MFMA 16x16x16 的 A `((16,4),4):((1,64),16)` 为例（tile 是 16 行）：`t % 16` 步长 1 → 行；`t // 16` 步长 64 = 4 x 16 → 每步 4 列；`v` 步长 16 → 每步 1 列。合起来就是 `行 = t % 16`、`K = 4*(t // 16) + v`。

下面把这四条规则写成一个小工具 `read_tv`：逐维列出"哪个坐标、走哪个方向、一步多远"，拼出公式，再对所有 `(t, v)` 用 `crd2idx` 验证一遍：

In [4]:
def leaves(shape, stride):
    """把嵌套的 shape / stride 展平成 [(大小, 步长), ...]，第一个最快"""
    if isinstance(shape, tuple):
        return [x for s, d in zip(shape, stride) for x in leaves(s, d)]
    return [(shape, stride)]


def read_tv(title, layout, tile_m, n_thr, n_val, col="K"):
    shape, stride = fx.get_shape(layout).to_py_value(), fx.get_stride(layout).to_py_value()
    print(f"{title}\n  layout = {layout}")
    terms = {"行": [], col: []}
    for var, name, s_mode, d_mode in [("t", "线程", shape[0], stride[0]), ("v", "值  ", shape[1], stride[1])]:
        lv = [x for x in leaves(s_mode, d_mode) if x[0] > 1]             # 大小为 1 的维只是占位
        P = 1
        for i, (s, d) in enumerate(lv):
            last = i == len(lv) - 1
            expr = (var if P == 1 and last else f"{var} % {s}" if P == 1 else
                    f"{var} // {P}" if last else f"({var} // {P}) % {s}")
            P *= s
            if d == 0:
                print(f"    {name} {expr:14} 大小 {s:<3} 步长 {d:<4} → 不影响位置（广播）")
                continue
            axis, step = ("行", d) if d < tile_m else (col, d // tile_m)
            terms[axis].append(expr if step == 1 else f"{step}*{expr}" if " " not in expr else f"{step}*({expr})")
            print(f"    {name} {expr:14} 大小 {s:<3} 步长 {d:<4} → {axis} +{step}")
    f_row, f_col = (" + ".join(terms[k]) or "0" for k in ["行", col])
    ok = all((o % tile_m, o // tile_m) == (eval(f_row, {"t": t, "v": v}), eval(f_col, {"t": t, "v": v}))
             for t in range(n_thr) for v in range(n_val) for o in [fx.crd2idx((t, v), layout).to_py_value()])
    print(f"  ⇒ 行 = {f_row}，{col} = {f_col}    （逐个 (t, v) 验证：{'一致' if ok else '不一致'}）\n")


with ir.Context() as ctx, ir.Location.unknown():
    ctx.load_all_available_dialects()
    with ir.InsertionPoint(ir.Module.create().body):
        atom = fx.make_mma_atom(fx.rocdl.MFMA(16, 16, 16, fx.BFloat16))
        read_tv("① MFMA 16x16x16 的 A（tile 16 x 16）", atom.layout_A_tv, 16, 64, 4)
        # → 行 = t % 16，K = 4*(t // 16) + v：就是 v0 的 row / kg
        read_tv("② MFMA 16x16x16 的 C（tile 16 x 16）", atom.layout_C_tv, 16, 64, 4, col="N")
        # → 行 = 4*(t // 16) + v，N = t % 16
        perm = fx.make_tiled_mma(atom, fx.make_layout((1, 1, 1), (0, 0, 0)),
                                 fx.make_tile(None, None, fx.make_layout((4, 4, 2), (1, 8, 4))))
        read_tv("③ 加 K 置换后的 A（tile 16 x 32）", perm.tv_layout_A_tiled, 16, 64, 8)
        # → K = 8*(t // 16) + v % 4 + 4*(v // 4) = 8*(t // 16) + v：每个 lane 8 个连续 K
        waves = fx.make_tiled_mma(atom, fx.make_layout((2, 2, 1), (1, 2, 0)))
        read_tv("④ 2x2 个 wave 的 C（tile 32 x 32）", waves.tv_layout_C_tiled, 32, 256, 4, col="N")
        # → (t // 16) % 8 里低 2 位是 lane 组、第 3 位是 wave % 2（行 +16），t // 128 是 wave // 2（列 +16）
        cp = fx.make_copy_atom(fx.UniversalCopy128b(), fx.BFloat16)
        tc = fx.make_tiled_copy_tv(cp, fx.make_layout((32, 8), (8, 1)), fx.make_layout((1, 8), (1, 1)))
        read_tv("⑤ make_tiled_copy_tv（tile 32 x 64）", tc.layout_tv_tiled, 32, 256, 8, col="列")
        # → 行 = t // 8，列 = 8*(t % 8) + v：每行 8 个线程，每个 8 个连续元素
        a32 = fx.make_mma_atom(fx.rocdl.MFMA(32, 32, 8, fx.BFloat16))
        read_tv("⑥ MFMA 32x32x8 的 C（tile 32 x 32）", a32.layout_C_tv, 32, 64, 16, col="N")
        # → 行 = 4*(t // 32) + v % 4 + 8*(v // 4)，N = t % 32

① MFMA 16x16x16 的 A（tile 16 x 16）
  layout = Layout<((16,4),4):((1,64),16)>
    线程 t % 16         大小 16  步长 1    → 行 +1
    线程 t // 16        大小 4   步长 64   → K +4
    值   v              大小 4   步长 16   → K +1
  ⇒ 行 = t % 16，K = 4*(t // 16) + v    （逐个 (t, v) 验证：一致）

② MFMA 16x16x16 的 C（tile 16 x 16）
  layout = Layout<((16,4),(4,1)):((16,4),(1,16))>
    线程 t % 16         大小 16  步长 16   → N +1
    线程 t // 16        大小 4   步长 4    → 行 +4
    值   v              大小 4   步长 1    → 行 +1
  ⇒ 行 = 4*(t // 16) + v，N = t % 16    （逐个 (t, v) 验证：一致）

③ 加 K 置换后的 A（tile 16 x 32）
  layout = Layout<((16,4),(4,(1,2))):((1,128),(16,(0,64)))>
    线程 t % 16         大小 16  步长 1    → 行 +1
    线程 t // 16        大小 4   步长 128  → K +8
    值   v % 4          大小 4   步长 16   → K +1
    值   v // 4         大小 2   步长 64   → K +4
  ⇒ 行 = t % 16，K = 8*(t // 16) + v % 4 + 4*(v // 4)    （逐个 (t, v) 验证：一致）

④ 2x2 个 wave 的 C（tile 32 x 32）
  layout = Layout<((16,8,2),((4,1),(1,1))):((32,4,512),((1,0),(0,0)))>
    线程 t % 16      

读 TV layout 的顺序总结一下：先分开线程和值两个维度 → 把嵌套的维度展平成"第一个最快"的坐标 → 步长和 tile 行数比较，判断沿行还是沿列、一步多远。③ 里的 `(1,2):(0,64)` 就是置换的效果：值的第 4～7 个跳到 K+4，和前 4 个拼成 8 个连续 K。④ 里 wave 的编号没有单独成一维，而是和 lane 组合并在 `(t // 16) % 8` 里，这也是这类串不好直接看懂的原因之一。

`partition_S` / `make_fragment_*` 的结果不是 TV layout，而是**一个线程自己的**那份 tensor，读法更简单：第 0 维是一次 atom 调用的值，后面各维是重复次数。例如 1.4 节的 `tcA.partition_S(gA)` = `((8,1),2,1):((1,0),512,0)`：一次 copy 8 个连续 bf16（步长 1），M 方向重复 2 次、相隔 512 个元素（16 行 x 32），K 方向只有 1 次。另外两个常见记号：`?` 是运行时才知道的值（`?{div=8}` 表示已知是 8 的倍数），`S<3,3,3> o ...` 是叠加了 swizzle 的 composed layout（[LDS_101](lds_101.ipynb)）。

下面把它扩展成完整的 GEMM，和手写的 v0 对照。

## 2. 同一个 kernel 的两种写法

先看最简单的一级：1 个 wave 算 `BM x BN` 的 C，每轮 K=32 直接从 global 读 A/B 进寄存器，再做 MFMA——就是 [v0.py](../kernels/gemm/v0.py)。下面是 tiled 写法，注释里对照 v0 的对应部分：

In [5]:
def build_tiled(BM, BN, M, N, K, mfma=(16, 16, 16), order=None, b_first=False):
    BK = 32
    k_per_lane = mfma[2] * mfma[0] // 64              # 一条 MFMA 里每个 lane 的 K 个数（16x16x16 是 4）

    @flyc.kernel(known_block_size=(64, 1, 1))
    def gemm_tiled(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor):
        tid = fx.thread_idx.x
        bm, bn = fx.block_idx.x, fx.block_idx.y
        # 本 block 的 A (BM, BK, K/BK)、B (BN, BK, K/BK)、C (BM, BN)
        # v0：make_ordered_layout((8, K//8, 16, BM//16, grid_m)) + 下标
        gA = fx.flat_divide(fx.make_view(fx.get_iter(A), fx.make_layout((M, K), (K, 1))), (BM, BK))[None, None, bm, None]
        gB = fx.flat_divide(fx.make_view(fx.get_iter(B), fx.make_layout((N, K), (K, 1))), (BN, BK))[None, None, bn, None]
        gC = fx.flat_divide(fx.make_view(fx.get_iter(C), fx.make_layout((M, N), (N, 1))), (BM, BN))[None, None, bm, bn]

        # lane → 元素的对应关系，v0：row = tid % 16，kg = tid // 16
        atom = fx.make_mma_atom(fx.rocdl.MFMA(*mfma, fx.BFloat16))
        if fx.const_expr(k_per_lane < 8):
            # v0 的技巧：每 lane 读 8 个连续 K（16 B）喂两条 MFMA，A/B 用同样的 K 置换
            groups = 64 // mfma[0]
            perm = fx.make_tile(None, None, fx.make_layout((k_per_lane, groups, 8 // k_per_lane), (1, 8, k_per_lane)))
            tiled_mma = fx.make_tiled_mma(atom, fx.make_layout((1, 1, 1), (0, 0, 0)), perm)
        else:
            tiled_mma = fx.make_tiled_mma(atom, fx.make_layout((1, 1, 1), (0, 0, 0)))
        thr_mma = tiled_mma.thr_slice(tid)

        # 搬运方式从 tiled_mma 推出来；v0：A[None, kk, row, i].load()
        cp = fx.make_copy_atom(fx.UniversalCopy128b(), fx.BFloat16)
        cpC = fx.make_copy_atom(fx.UniversalCopy32b(), fx.Float32)
        tcA = fx.make_tiled_copy_A(cp, tiled_mma).get_slice(tid)
        tcB = fx.make_tiled_copy_B(cp, tiled_mma).get_slice(tid)
        tcC = fx.make_tiled_copy_C(cpC, tiled_mma).get_slice(tid)
        tAg, tBg = tcA.partition_S(gA), tcB.partition_S(gB)       # 本线程要读的 global 元素
        fA = thr_mma.make_fragment_A(gA[None, None, 0])            # 寄存器里的 MFMA 操作数
        fB = thr_mma.make_fragment_B(gB[None, None, 0])
        fC = thr_mma.make_fragment_C(gC)                           # v0：acc 列表 + init/yield
        fC.fill(0)
        rA, rB = tcA.retile(fA), tcB.retile(fB)                    # 同一块寄存器，按 copy 的形状来看

        for kt in range(fx.Int32(0), fx.Int32(K // BK), fx.Int32(1)):
            if fx.const_expr(b_first):                             # 4.2 节用：先发 B 的 load
                fx.copy(cp, tBg[None, None, None, kt], rB)
            fx.copy(cp, tAg[None, None, None, kt], rA)
            if fx.const_expr(not b_first):
                fx.copy(cp, tBg[None, None, None, kt], rB)
            fx.gemm(atom, fC, fA, fB, fC, traversal_order=order)  # v0：三层循环 + shuffle + mfma
        fx.copy(cpC, tcC.retile(fC), tcC.partition_S(gC))         # v0：按 C 的排布逐个标量写

    @flyc.jit
    def launch(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor, stream: fx.Stream):
        gemm_tiled(A, B, C).launch(grid=(M // BM, N // BN, 1), block=(64, 1, 1), stream=stream)

    return launch


M = N = K = 512
A = torch.randn(M, K, dtype=torch.bfloat16, device="cuda")
B = torch.randn(N, K, dtype=torch.bfloat16, device="cuda")
C = torch.zeros(M, N, device="cuda")
build_tiled(64, 64, M, N, K)(A, B, C, stream)
torch.cuda.synchronize()
print("本机 GPU 正确:", torch.allclose(C, A.float() @ B.float().T, atol=1e-2, rtol=1e-2))   # → True

本机 GPU 正确: True


| 要做的事 | v0 手写 | tiled |
|---|---|---|
| 切出本 block 的 A/B/C | `make_ordered_layout((8, K//8, 16, BM//16, grid))`，每一维的含义自己记 | `flat_divide(gA, (BM, BK))[None, None, bm, None]`，保持 (M, K) 的逻辑坐标 |
| 每个 lane 负责哪些元素 | `row = lane % 16`、`kg = lane // 16`，来自 MFMA 文档 | `tiled_mma` 里的 MFMA atom 自带 |
| 每 lane 读 16 B | 8 个连续 K 拆给两条 MFMA，A/B 置换方式要一致 | `make_tiled_mma` 的 permutation 参数 |
| 读 A/B | 自己写下标 `A[None, kk, row, i].load()` | `fx.copy(atom, partition_S(gA)[..., kt], retile(fA))` |
| MFMA | 三层循环 + `shuffle` 取 4 个元素 + `mfma_f32_16x16x16bf16_1k` | `fx.gemm(atom, fC, fA, fB, fC)` |
| 累加器 | `Vector` 列表，用 `init=` / `yield` 作为 loop-carried 值 | 寄存器 tensor `fC`，就地更新 |
| 写 C | 自己推出 lane 持有 `C[4*(l//16)+r, l%16]`，逐个写 | `make_tiled_copy_C` + `fx.copy` |

### 2.1 tiled_mma 给出的排布就是 v0 手算的那个

`tiled_mma.tv_layout_A_tiled` 是 (线程, 值) → A tile 里偏移的 layout，可以在 host 上直接算（见 [layout_algebra](layout_algebra.ipynb)），和 v0 的公式逐个比较：

In [6]:
with ir.Context() as ctx, ir.Location.unknown():
    ctx.load_all_available_dialects()
    with ir.InsertionPoint(ir.Module.create().body):
        atom = fx.make_mma_atom(fx.rocdl.MFMA(16, 16, 16, fx.BFloat16))
        tiled = fx.make_tiled_mma(atom, fx.make_layout((1, 1, 1), (0, 0, 0)),
                                  fx.make_tile(None, None, fx.make_layout((4, 4, 2), (1, 8, 4))))
        LA, LC = tiled.tv_layout_A_tiled, tiled.tv_layout_C_tiled
        print("tile (M, N, K):", tiled.tile_size_mnk)                # → (16, 16, 32)
        print("A 的 TV layout:", LA)
        print("C 的 TV layout:", LC)
        mis_a = mis_c = 0
        for l in range(64):
            for v in range(8):                                       # A：每 lane 8 个值，tile 是 16 x 32
                o = fx.crd2idx((l, v), LA).to_py_value()
                mis_a += (o % 16, o // 16) != (l % 16, 8 * (l // 16) + v)        # v0：row = l%16，k = 8*kg + v
            for r in range(4):                                       # C：每 lane 4 个值，tile 是 16 x 16
                o = fx.crd2idx((l, r), LC).to_py_value()
                mis_c += (o % 16, o // 16) != (4 * (l // 16) + r, l % 16)        # v0：C[4*kg + r, row]
        print("与 v0 公式不一致的个数: A", mis_a, "| C", mis_c)        # → A 0 | C 0

tile (M, N, K): IntTuple<(16,16,32)>
A 的 TV layout: Layout<((16,4),(4,(1,2))):((1,128),(16,(0,64)))>
C 的 TV layout: Layout<((16,4),((4,1),(1,1))):((16,4),((1,0),(0,0)))>
与 v0 公式不一致的个数: A 0 | C 0


两种写法在 lane 和元素的对应关系上完全一样，区别只在谁来推导：手写版要自己从 MFMA 文档推出来并保证 A、B、C 三处一致；tiled 版由 atom 推出，排布本身也可以打印出来检查。

## 3. 生成的 ISA、编译时间、性能

v0（手写）和 tiled 版，`BM = BN = 64 / 128`。tiled 版比较 `fx.gemm` 的默认遍历顺序和 `MNK`（含义见 4.2 节）：

In [7]:
M = N = K = 4096
A = torch.randn(M, K, dtype=torch.bfloat16, device="cuda") / 64
B = torch.randn(N, K, dtype=torch.bfloat16, device="cuda") / 64
C = torch.zeros(M, N, device="cuda")
ref = A.float() @ B.float().T
measure(lambda: build_tiled(64, 64, 256, 256, 256), A[:256, :256].contiguous(), B[:256, :256].contiguous(),
        C[:256, :256].contiguous(), ref[:256, :256])        # 先编译一个小的，避免第一次编译的启动开销算进去

MNK = fx.GemmTraversalOrder.MNK
for bm in [64, 128]:
    cases = {"v0 手写": lambda: build_v0(bm, bm, 1, M, N, K),
             "tiled 默认顺序": lambda: build_tiled(bm, bm, M, N, K),
             "tiled MNK": lambda: build_tiled(bm, bm, M, N, K, order=MNK)}
    rows = {}
    for name, build in cases.items():
        s = isa_stats(build, A, B, C, stream)
        rows[name] = {**measure(build, A, B, C, ref), **{k: s["gfx942"][k] for k in
                      ["MFMA", "global_load", "循环 VALU", "循环指令", "总指令", "VGPR"]},
                      "gfx950 同": s["gfx942"] == s["gfx950"]}
    print(f"==== BM = BN = {bm}（ISA 统计是 gfx942 的，最后一列表示 gfx950 是否完全相同）")
    table(rows)
# → BM=64 ：三者都约 270 TFLOPS；MFMA 64、global_load_dwordx4 16 条都相同；循环 VALU 44 vs 18，总指令 464 vs 311
# → BM=128：v0 约 455，tiled 默认顺序约 345，tiled MNK 约 455 TFLOPS；编译 v0 约 1 s，tiled 约 0.4 s
print("代码行数：build_v0", lines(build_v0), "| build_tiled", lines(build_tiled))

==== BM = BN = 64（ISA 统计是 gfx942 的，最后一列表示 gfx950 是否完全相同）
               | 正确 | 编译 ms |    us | TFLOPS | MFMA | global_load | 循环 VALU | 循环指令 | 总指令 | VGPR | gfx950 同
v0 手写        | True |     126 | 528.5 |  260.1 |   64 |          16 |        44 |      145 |    464 |  112 |      True
tiled 默认顺序 | True |      83 | 505.9 |  271.7 |   64 |          16 |        18 |      116 |    311 |  148 |      True
tiled MNK      | True |      81 | 510.8 |  269.1 |   64 |          16 |        18 |      116 |    311 |  144 |      True
==== BM = BN = 128（ISA 统计是 gfx942 的，最后一列表示 gfx950 是否完全相同）
               | 正确 | 编译 ms |    us | TFLOPS | MFMA | global_load | 循环 VALU | 循环指令 | 总指令 | VGPR | gfx950 同
v0 手写        | True |     963 | 300.9 |  456.7 |  128 |          16 |        56 |      217 |   1407 |  336 |      True
tiled 默认顺序 | True |     399 | 397.5 |  345.8 |  128 |          16 |        34 |      203 |    969 |  328 |      True
tiled MNK      | True |     411 | 303.7 |  452.6 |  128 |          16 |       

- **指令选择相同**：同样的 `global_load_dwordx4` 和 `v_mfma_f32_16x16x16_bf16` 条数，gfx942 / gfx950 生成的 ISA 也完全相同。
- **地址计算更少**：v0 循环里每个 tile 单独算地址（`v_or_b32`、`v_ashrrev_i32`、`v_lshl_add_u64`），tiled 版大多是"同一个指针 + 常数"，循环 VALU 少一半多。epilogue 也一样：v0 逐个标量写 C，每次都算一次地址，总指令多 40%～50%。原因见 4.1 节。
- **编译更快**：v0 用 Python 列表展开出很多独立的 `Vector` 操作和 loop-carried 值，tiled 版的 IR 更紧凑，`BM=128` 时编译时间不到一半。
- **遍历顺序是性能开关**：`fx.gemm` 默认顺序下，`BM=128` 比 v0 慢约 25%；换成 `MNK` 就和 v0 持平（`KMN`、`MKN`、`MNK` 及其 serpentine 变体都快，`KNM`、`NKM`、`NMK` 和默认值慢）。原因见 4.2 节。
- 代码行数相近：这一级的 kernel 本来就短，tiled 的优势不在行数，而在于不用自己推导和维护排布。

## 4. 深入：地址计算和遍历顺序

### 4.1 为什么 tiled 版循环里的地址计算少

先看 `BM = 64` 时 A 的 4 个 load 在 LLVM IR（`21_llvm_ir.ll`，生成 ISA 之前）里的样子：

In [8]:
def a_loads(build, n=4):
    """LLVM IR 里前 n 个 <8 x bfloat> load（都是 A）及其地址计算"""
    ll = dumps(build(), A, B, C, stream, arch="gfx942")["21_llvm_ir.ll"]
    lines = [l.strip() for l in ll.splitlines()]
    defs = {l.split(" = ")[0]: l for l in lines if re.match(r"%\d+ = ", l)}
    out, count = [], 0
    for l in lines:
        m = re.match(r"%\d+ = load <8 x bfloat>, ptr addrspace\(1\) (%\d+)", l)
        if not m:
            continue
        gep = defs[m.group(1)]
        idx = defs.get(gep.split()[-1])                       # 下标是 SSA 值时，它的定义
        if idx:
            consts = [defs[x] for x in re.findall(r"%\d+", idx.split(" = ")[1])
                      if x in defs and re.search(r"add i32 %\d+, \d{5,}", defs[x])]
            out += consts + [idx]
        out += [gep, l, ""]
        count += 1
        if count == n:
            return out


for name, build in [("v0", lambda: build_v0(64, 64, 1, M, N, K)), ("tiled", lambda: build_tiled(64, 64, M, N, K, order=MNK))]:
    print(f"==== {name}")
    for l in a_loads(build):
        print("   ", l[:110])
# → v0：每个 load 都是 getelementptr %0, i32 <完整下标>，下标 = 动态部分 + 65536 / 131072 / 196608（i32 加法）
# → tiled：只有第一个用 %0 + 动态下标，后三个都是 getelementptr <第一个指针>, i32 65536 / 131072 / 196608

==== v0
    %55 = add i32 %16, %54
    %56 = getelementptr bfloat, ptr addrspace(1) %0, i32 %55
    %57 = load <8 x bfloat>, ptr addrspace(1) %56, align 2
    
    %59 = add i32 %54, 65536
    %60 = add i32 %16, %59
    %61 = getelementptr bfloat, ptr addrspace(1) %0, i32 %60
    %62 = load <8 x bfloat>, ptr addrspace(1) %61, align 2
    
    %64 = add i32 %54, 131072
    %65 = add i32 %16, %64
    %66 = getelementptr bfloat, ptr addrspace(1) %0, i32 %65
    %67 = load <8 x bfloat>, ptr addrspace(1) %66, align 2
    
    %69 = add i32 %54, 196608
    %70 = add i32 %16, %69
    %71 = getelementptr bfloat, ptr addrspace(1) %0, i32 %70
    %72 = load <8 x bfloat>, ptr addrspace(1) %71, align 2
    
==== tiled
    %33 = add i32 %25, %32
    %34 = getelementptr bfloat, ptr addrspace(1) %0, i32 %33
    %35 = load <8 x bfloat>, ptr addrspace(1) %34, align 16
    
    %36 = getelementptr bfloat, ptr addrspace(1) %34, i32 65536
    %37 = load <8 x bfloat>, ptr addrspace(1) %36, align 16
    
  

差别在加常数的位置：

- v0 每次用 `A[None, kk, row, i]` 从整个大 view 取元素，layout 把坐标算成一个 **i32 下标**（动态部分 + `i * 16 * K`），再从矩阵起始地址取地址。地址是 64 位的，i32 下标要先符号扩展；而 `sext(x + c)` 不一定等于 `sext(x) + c`（i32 加法可能溢出，这里的 `add` 没有 `nsw` 标记），LLVM 不能把常数提到 64 位地址上，只好每个 load 都重新做一遍"i32 加法 → 符号扩展 → 乘 2 加基址"。
- tiled 版的 `partition_S` 先把和线程、`kt` 有关的动态部分算进了 tensor 的**指针**，剩下的重复维度都是静态 stride，`fx.copy` 对它们生成"同一个指针 + 常数"。指针加常数在 64 位上做，不存在符号扩展的问题。

反映到 ISA（只看循环开头到第 4 个 load，省略 MFMA 和 `s_nop`）：

In [9]:
def loop_head(build, n_loads=4):
    lp = main_loop(body(isa(build(), A, B, C, stream, arch="gfx942")))
    out = []
    for l in lp[1:]:
        if l.startswith(("s_nop", "v_mfma")):
            continue
        out.append(l)
        if sum(x.startswith("global_load") for x in out) == n_loads:
            return out


for name, build in [("v0", lambda: build_v0(64, 64, 1, M, N, K)), ("tiled", lambda: build_tiled(64, 64, M, N, K, order=MNK))]:
    print(f"==== {name}")
    for l in loop_head(build):
        print("   ", l)
# → v0：每个 load 前各有一组 v_or_b32（i32 加常数）+ v_ashrrev_i32（符号扩展）+ v_lshl_add_u64（x2 + 基址）
# → tiled：A、B 各一组 v_lshl_add_u64 算出基地址，其余 load 的地址是 v_add_co / v_addc（基地址 + SGPR 里的常数）

==== v0
    v_lshl_add_u32 v66, s12, 5, v69
    v_or_b32_e32 v70, s4, v66
    v_ashrrev_i32_e32 v71, 31, v70
    v_or_b32_e32 v75, 0x10000, v66
    v_or_b32_e32 v82, 0x20000, v66
    s_waitcnt lgkmcnt(0)
    v_lshl_add_u64 v[70:71], v[70:71], 1, s[6:7]
    v_or_b32_e32 v83, 0x30000, v66
    v_or_b32_e32 v72, s10, v66
    v_or_b32_e32 v74, s4, v75
    v_or_b32_e32 v78, s10, v82
    global_load_dwordx4 v[84:87], v[70:71], off
    v_ashrrev_i32_e32 v73, 31, v72
    v_or_b32_e32 v76, s10, v75
    v_or_b32_e32 v80, s10, v83
    v_ashrrev_i32_e32 v75, 31, v74
    v_ashrrev_i32_e32 v79, 31, v78
    v_lshl_add_u64 v[72:73], v[72:73], 1, s[8:9]
    v_ashrrev_i32_e32 v77, 31, v76
    v_ashrrev_i32_e32 v81, 31, v80
    v_lshl_add_u64 v[70:71], v[74:75], 1, s[6:7]
    v_lshl_add_u64 v[74:75], v[78:79], 1, s[8:9]
    global_load_dwordx4 v[88:91], v[72:73], off
    global_load_dwordx4 v[96:99], v[74:75], off
    v_lshl_add_u64 v[72:73], v[76:77], 1, s[8:9]
    v_lshl_add_u64 v[76:77], v[80:81], 1, s

这不是 tiled 独有的能力。手写版只要**先用动态坐标切出子 tensor，再用静态下标**，就能得到同样的"指针 + 常数"。把 v0 里的两行改成这样（`v0_slice`）：

In [10]:
V0_SRC = open(sys.modules["v0"].__file__).read()
PATCH_DIR = tempfile.mkdtemp()
sys.path.insert(0, PATCH_DIR)


def patched_v0(name, *replacements):
    """把 v0.py 的源码按 (旧, 新) 替换后存成模块 name，返回其中的 build 函数"""
    src = V0_SRC.replace("def build_v0(", f"def build_{name}(")
    for old, new in replacements:
        assert old in src
        src = src.replace(old, new)
    with open(os.path.join(PATCH_DIR, f"{name}.py"), "w") as f:
        f.write(src)
    return getattr(__import__(name), f"build_{name}")


SLICE = [("a_tile = [A[None, kk, row, i].load()",
          "Ak, Bk = A[None, kk, row, None], B[None, kk, row, None]   # 先切动态坐标\n"
          "            a_tile = [Ak[None, i].load()"),
         ("b_tile = [B[None, kk, row, j].load()", "b_tile = [Bk[None, j].load()")]
build_v0_slice = patched_v0("v0_slice", *SLICE)
print("\n".join(l for l in inspect.getsource(build_v0_slice).splitlines() if "Ak" in l or "Bk" in l))

for bm in [64, 128]:
    rows = {}
    for name, build in [("v0", lambda: build_v0(bm, bm, 1, M, N, K)), ("v0_slice", lambda: build_v0_slice(bm, bm, 1, M, N, K)),
                        ("tiled MNK", lambda: build_tiled(bm, bm, M, N, K, order=MNK))]:
        s = isa_stats(build, A, B, C, stream)["gfx942"]
        r = measure(build, A, B, C, ref)
        rows[name] = {"正确": r["正确"], "TFLOPS": r["TFLOPS"], **{k: s[k] for k in ["循环 VALU", "循环指令", "VGPR"]}}
    print(f"==== BM = BN = {bm}")
    table(rows)
# → BM=128：循环 VALU v0 56 → v0_slice 35，和 tiled 的 34 相当；但 v0_slice 的 TFLOPS 反而从约 455 掉到约 350（4.2 节解释）
# → BM=64 ：v0_slice 35，tiled 18，但 tiled 的 VGPR 多（144 vs 106）；三者 TFLOPS 相同

            Ak, Bk = A[None, kk, row, None], B[None, kk, row, None]   # 先切动态坐标
            a_tile = [Ak[None, i].load().bitcast(fx.Int16) for i in range_constexpr(NI)]
            b_tile = [Bk[None, j].load().bitcast(fx.Int16) for j in range_constexpr(NJ)]
==== BM = BN = 64
          | 正确 | TFLOPS | 循环 VALU | 循环指令 | VGPR
v0        | True |  271.3 |        44 |      145 |  112
v0_slice  | True |  271.8 |        35 |      144 |  106
tiled MNK | True |  268.5 |        18 |      116 |  144
==== BM = BN = 128
          | 正确 | TFLOPS | 循环 VALU | 循环指令 | VGPR
v0        | True |  456.0 |        56 |      217 |  336
v0_slice  | True |  346.8 |        35 |      201 |  330
tiled MNK | True |  450.2 |        34 |      207 |  328


- `BM=128` 时，先切动态坐标就把 v0 的循环 VALU 降到和 tiled 一样。但它反而慢了约 25%——这和 tiled 默认顺序慢是同一个原因，见 4.2 节。
- `BM=64` 时还差一半，剩下的差别来自寄存器分配：LLVM 把这个循环展开成两轮，tiled 版的 8 个地址（16 个 VGPR）在两轮间一直保留，第二轮直接用 `offset:64`（下一个 K=32 块）；v0_slice 的寄存器少，第二轮重新算一遍地址。是用 VGPR 换 VALU 的取舍，而不是写法本身的差别。
- 这个 kernel 是 MFMA 受限的，这些 VALU 和 MFMA 并行执行，少算地址本身不提高性能。当 VALU 本身很忙（dequant、softmax、大量地址计算）或者 VGPR 紧张时，这个差别才会反映到性能上。

**结论**：地址计算少不是因为 tiled 更"聪明"，而是 partition 自然地把动态部分放进指针、把静态部分留给常数偏移。手写时记住同一条规则：动态坐标先切，静态下标后取。

### 4.2 为什么 MNK 比默认顺序快

先搞清楚名字的含义。`BM = BN = 32`（2x2 个 MFMA 块、每块 2 条 MFMA 覆盖 K=32）时，看 LLVM IR 里 MFMA 调用的顺序（还没经过指令调度）。记号：`A0·B1` 表示用第 0 个 A 操作数和第 1 个 B 操作数（按第一次出现编号），`+` 表示累加在前面某条 MFMA 的结果上（即 K 方向的第二条）：

In [11]:
MFMA_CALL = re.compile(r"(%\S+) = (?:tail )?call <4 x float> @llvm\.amdgcn\.mfma\.f32\.16x16x16bf16\.1k\("
                       r"(?:<4 x i16> )?(%\S+), (?:<4 x i16> )?(%\S+), (?:<4 x float> )?(%\S+),")


def mfma_order(order):
    ll = dumps(build_tiled(32, 32, M, N, K, order=order), A, B, C, stream, arch="gfx942")["21_llvm_ir.ll"]
    ids, results, seq = {}, set(), []
    name = lambda kind, v: ids.setdefault((kind, v), f"{kind}{sum(k == kind for k, _ in ids)}")
    for res, a, b, acc in MFMA_CALL.findall(ll):
        seq.append(f"{name('A', a)}·{name('B', b)}" + ("+" if acc in results else ""))
        results.add(res)
    first, second = seq[0], seq[1]
    inner = ("K" if second.endswith("+") else
             "M" if first.split("·")[1] == second.split("·")[1].rstrip("+") else "N")
    return seq, inner


O = fx.GemmTraversalOrder
for order in [None, O.MNK, O.NMK, O.KMN, O.MKN]:
    seq, inner = mfma_order(order)
    print(f"{str(order):5} 最内层 {inner}: {' '.join(seq)}")
# → MNK：A0·B0 A1·B0 A0·B1 A1·B1 …：B 不变、A 先变 → M 最内，N 其次，K 最外
# → 名字里第一个字母是最内层；默认顺序和 NMK 相同（N 最内）

None  最内层 N: A0·B0 A0·B1 A1·B0 A1·B1 A2·B2+ A2·B3+ A3·B2+ A3·B3+
mnk   最内层 M: A0·B0 A1·B0 A0·B1 A1·B1 A2·B2+ A3·B2+ A2·B3+ A3·B3+
nmk   最内层 N: A0·B0 A0·B1 A1·B0 A1·B1 A2·B2+ A2·B3+ A3·B2+ A3·B3+
kmn   最内层 K: A0·B0 A1·B1+ A2·B0 A3·B1+ A0·B2 A1·B3+ A2·B2 A3·B3+
mkn   最内层 M: A0·B0 A1·B0 A2·B1+ A3·B1+ A0·B2 A1·B2 A2·B3+ A3·B3+


所以 `MNK` 是"M 最内层"：固定一个 B 块，先把所有 A 块和它乘完。默认顺序和 `NMK` 一样，是固定一个 A 块，先遍历所有 B 块。

再看 `BM = 128` 的主循环。每轮发 16 条 load（8 个 A 块、8 个 B 块），然后是 128 条 MFMA。下面统计两件事：

- load 的发出顺序（按它写入的寄存器被 MFMA 当作 A 还是 B 用来判断）；
- 相邻两次 `s_waitcnt vmcnt` 之间有几条 MFMA。

除了 tiled 的两种顺序，还包括 v0、v0_slice，以及把 v0_slice 的 MFMA 循环从 `for i (M): for j (N)` 换成 `for j: for i` 的 `v0_slice_ji`（M 变成内层）：

In [12]:
def reg_set(s):
    m = re.match(r"([av])\[(\d+):(\d+)\]", s) or re.match(r"([av])(\d+)()", s)
    lo = int(m.group(2))
    return {(m.group(1), r) for r in range(lo, int(m.group(3) or lo) + 1)}


def load_profile(build):
    lp = main_loop(body(isa(build(), A, B, C, stream, arch="gfx942")))
    loads, kinds, gaps, n = [], [], [], 0
    for l in lp:
        if l.startswith("global_load"):
            loads.append(reg_set(l.split()[1].rstrip(",")))
            kinds.append("?")
        elif l.startswith("v_mfma"):
            ops = [x.strip() for x in l.split(None, 1)[1].split(",")]
            for i, r in enumerate(loads):
                if kinds[i] == "?":
                    kinds[i] = "A" if reg_set(ops[1]) & r else "B" if reg_set(ops[2]) & r else "?"
            n += 1
        elif re.match(r"s_waitcnt vmcnt", l):
            gaps.append(n)
            n = 0
    return "".join(kinds), gaps[1:]


SWAP_IJ = ("            for i in range_constexpr(NI):\n                for j in range_constexpr(NJ):\n",
           "            for j in range_constexpr(NJ):\n                for i in range_constexpr(NI):\n")
build_v0_slice_ji = patched_v0("v0_slice_ji", *SLICE, SWAP_IJ)
for name, build in [("v0", lambda: build_v0(128, 128, 1, M, N, K)),
                    ("v0_slice", lambda: build_v0_slice(128, 128, 1, M, N, K)),
                    ("v0_slice_ji", lambda: build_v0_slice_ji(128, 128, 1, M, N, K)),
                    ("tiled 默认", lambda: build_tiled(128, 128, M, N, K)),
                    ("tiled MNK", lambda: build_tiled(128, 128, M, N, K, order=MNK))]:
    kinds, gaps = load_profile(build)
    print(f"{name}{' ' * (12 - width(name))}{measure(build, A, B, C, ref)['TFLOPS']:6} TFLOPS | load 顺序 {kinds} | 两次等待之间的 MFMA 数 {gaps}")
# → v0         约 455：ABABABAB…（调度器把 A、B 交替发出）
# → v0_slice   约 345：AAAAAAAABBBBBBBB，等待间隔 [1, 1, 1, …]
# → v0_slice_ji 约 455～470：AAAAAAAABBBBBBBB，等待间隔 [8, 8, 8, …]
# → tiled 默认  约 345：AAAAAAAABBBBBBBB，等待间隔 [1, 1, 1, …]
# → tiled MNK  约 455：AAAAAAAABBBBBBBB，等待间隔 [8, 8, 8, …]

v0           456.9 TFLOPS | load 顺序 ABABABABABABABAB | 两次等待之间的 MFMA 数 [1, 1, 1, 2, 4, 1, 1, 3, 1]
v0_slice     346.0 TFLOPS | load 顺序 AAAAAAAABBBBBBBB | 两次等待之间的 MFMA 数 [1, 1, 1, 1, 1, 1, 1]
v0_slice_ji  461.5 TFLOPS | load 顺序 AAAAAAAABBBBBBBB | 两次等待之间的 MFMA 数 [8, 8, 8, 8, 8, 8, 8]
tiled 默认   345.9 TFLOPS | load 顺序 AAAAAAAABBBBBBBB | 两次等待之间的 MFMA 数 [1, 1, 1, 1, 1, 1, 1]
tiled MNK    446.6 TFLOPS | load 顺序 AAAAAAAABBBBBBBB | 两次等待之间的 MFMA 数 [8, 8, 8, 8, 8, 8, 8]


`fx.copy` 先发 A 的 8 条 load，再发 B 的 8 条。`vmcnt` 按发出顺序计数（[waitcnt_101](waitcnt_101.ipynb)），所以数据也大致按 A0…A7、B0…B7 的顺序到达：

- **MNK（M 最内）**：第一条 MFMA 等到 B0，然后 B0 和所有 A 块做 8 条 MFMA；这期间 B1 到达，再做 8 条……每等一条新 load，有 8 条 MFMA 的计算可以掩盖下一条的延迟。
- **默认（N 最内）**：A0 要依次和 B0…B7 相乘，而 B 是最后发出的一组，几乎每条 MFMA 前都要等一条新的 load。调度器还把一部分 B 的 load 插到了 MFMA 之间，每条 MFMA 前都是 `vmcnt(1)`：等的就是上一条 MFMA 之前才发出的那个 load，几乎完整地暴露了一次访存延迟。
- **v0 的三个版本**说明这和写法无关：v0 的代码顺序是 N 在 M 里面（和默认顺序一样），原版快只是因为每个 load 有自己独立的地址计算，调度器恰好把 A、B 交替发出了。v0_slice 地址算得少了，load 按代码顺序集中发出（先 A 后 B），就和 tiled 默认顺序一样慢；再把循环换成 M 在内（v0_slice_ji），又回到和 v0、tiled MNK 一样的水平。

如果这个解释对，把 A、B 的 copy 顺序对调（先发 B），快慢就应该反过来：

In [13]:
rows = {}
for order in [None, O.MNK, O.NMK, O.KMN, O.KNM]:
    rows[str(order)] = {f"先发 {first}": measure(lambda: build_tiled(128, 128, M, N, K, order=order, b_first=first == "B"),
                                                A, B, C, ref)["TFLOPS"] for first in "AB"}
table(rows)
# → 先发 A：None / NMK / KNM 约 350，MNK / KMN 约 450
# → 先发 B：正好反过来

     | 先发 A | 先发 B
None |  342.5 |  452.0
mnk  |  455.1 |  345.4
nmk  |  348.1 |  452.6
kmn  |  444.8 |  345.6
knm  |  354.8 |  450.8


结果完全反过来了。所以没有"本身更快"的遍历顺序，关键是 **MFMA 消费操作数的顺序要和 load 发出的顺序匹配**：变化最快的（最内层）应该是先发出的那个操作数，最后发出的操作数变化最慢。一般做法：

- 直接从 global 读、没有预取时，按上面的规则搭配 copy 顺序和 `traversal_order`，或者干脆试一遍几种顺序；
- 更根本的办法是预取（第 6 节的 `prefetch`、v2 的流水）：让 MFMA 用的是上一轮就已到达的数据，顺序的影响就小得多（第 6 节带 LDS 的版本里，默认顺序和 MNK 只差约 1%）。

## 5. 换一条 MFMA 指令

tiled 版换 MFMA 只改 atom 一个参数，搬运、置换、写回全部跟着变。gfx950 新增的 `16x16x32` / `32x32x16` bf16 每 lane 本来就是 8 个连续 K，不再需要置换（`k_per_lane >= 8` 分支）。

注意：在 gfx942 上编译这两条 gfx950 专有指令，LLVM 直接 abort（`Cannot select: intrinsic %llvm.amdgcn.mfma.f32.16x16x32.bf16`），整个进程退出，所以下面只对 gfx950 编译它们。

In [14]:
rows = {}
for mfma in [(16, 16, 16), (32, 32, 8), (16, 16, 32), (32, 32, 16)]:
    archs = ARCHS if mfma[2] * mfma[0] <= 256 else ("gfx950",)       # 后两条只有 gfx950 有
    build = lambda: build_tiled(128, 128, M, N, K, mfma=mfma, order=MNK)
    s = {arch: stats(isa(build(), A, B, C, stream, arch=arch)) for arch in archs}
    rows[str(mfma)] = {"可编译": "/".join(archs), "MFMA": s["gfx950"]["MFMA"],
                       "VGPR": s["gfx950"]["VGPR"], **measure(build, A, B, C, ref)}
table(rows)
# → 四种都正确；16x16x32 约 480 TFLOPS 最快，32x32x16 约 415，32x32x8 只有约 265

             |        可编译 | MFMA | VGPR | 正确 | 编译 ms |    us | TFLOPS
(16, 16, 16) | gfx942/gfx950 |  128 |  328 | True |     410 | 302.8 |  453.9
(32, 32, 8)  | gfx942/gfx950 |   64 |  512 | True |     305 | 517.3 |  265.7
(16, 16, 32) |        gfx950 |   64 |  328 | True |     320 | 284.3 |  483.4
(32, 32, 16) |        gfx950 |   64 |  500 | True |     291 | 328.6 |  418.2


四种 atom 都一次写对。手写版要换指令，得重写所有和排布有关的代码：每 lane 的行号和 K 分组、操作数怎么从 16 B 里切、C 的写回位置。以 `32x32x8` 为例，C 的排布是这样的：

In [15]:
with ir.Context() as ctx, ir.Location.unknown():
    ctx.load_all_available_dialects()
    with ir.InsertionPoint(ir.Module.create().body):
        for mfma in [(16, 16, 16), (32, 32, 8)]:
            atom = fx.make_mma_atom(fx.rocdl.MFMA(*mfma, fx.BFloat16))
            print(f"{str(mfma):12} A: {atom.layout_A_tv}   C: {atom.layout_C_tv}")
# → 16x16x16 的 C 每 lane 4 个值；32x32x8 每 lane 16 个值，分成 4 组、组间隔 8 行

(16, 16, 16) A: Layout<((16,4),4):((1,64),16)>   C: Layout<((16,4),(4,1)):((16,4),(1,16))>
(32, 32, 8)  A: Layout<((32,2),4):((1,128),32)>   C: Layout<((32,2),(4,4)):((32,4),(1,8))>


`32x32x8` 的 C：lane `l` 持有第 `l % 32` 列的 16 个值，行号是 `8*g + 4*(l//32) + r`（g、r = 0..3）。手写时这些都要自己推、自己验证；atom 已经把它们编码在 layout 里了。另外 `32x32x8` 在这里明显更慢，tile 配置没有为它调过；换指令时 tile 大小、遍历顺序也要跟着重新选。

## 6. 加上 LDS 和 swizzle（对应 v1）

v1：4 个 wave（2x2），CTA tile 256x256，每轮 K=64 先读进寄存器、写进 LDS（swizzle 避免 bank conflict），barrier 后从 LDS 读出做 MFMA。tiled 版：

- global → LDS：`make_tiled_copy_tv` 指定 256 个线程的排布（每行 8 个线程 x 16 B），`partition_S(gA)` / `partition_D(sA)`；
- LDS tensor 直接用 swizzle 的 composed layout（`Swizzle(3,3,3)`，和 v1 手写的 `chunk ^ (row % 8)` 相同），`fx.copy` 能读写这种 tensor（`.load()/.store()` 不行，见 [LDS_101](lds_101.ipynb) 第 5 节）；
- LDS → 寄存器：`make_tiled_copy_A/B(atom, tiled_mma)`；`tiled_mma` 的 atom 排布 `(2, 2, 1)` 就是 2x2 个 wave。

另外加两个开关：`prefetch` 把下一轮的 global 读提前到本轮 MFMA 之前（只是挪了两行 `fx.copy`），`split` 把每轮的 LDS 读和 `fx.gemm` 按 K=32 拆成两段（对切好的 tensor 再取下标即可）。

In [16]:
def build_tiled_lds(BM, BN, M, N, K, swizzle=True, prefetch=False, split=False, order=fx.GemmTraversalOrder.MNK):
    BK = 64

    @fx.struct
    class Smem:
        a: fx.Array[fx.BFloat16, BM * BK, 16]
        b: fx.Array[fx.BFloat16, BN * BK, 16]

    @flyc.kernel(known_block_size=(256, 1, 1))
    def gemm_tiled_lds(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor):
        tid = fx.thread_idx.x
        bm, bn = fx.block_idx.x, fx.block_idx.y
        gA = fx.flat_divide(fx.make_view(fx.get_iter(A), fx.make_layout((M, K), (K, 1))), (BM, BK))[None, None, bm, None]
        gB = fx.flat_divide(fx.make_view(fx.get_iter(B), fx.make_layout((N, K), (K, 1))), (BN, BK))[None, None, bn, None]
        gC = fx.flat_divide(fx.make_view(fx.get_iter(C), fx.make_layout((M, N), (N, 1))), (BM, BN))[None, None, bm, bn]
        lds = fx.SharedAllocator().allocate(Smem).peek()

        def smem(ptr, rows):
            layout = fx.make_ordered_layout((rows, BK), (1, 0))
            if fx.const_expr(swizzle):              # v1：lds_chunk(L, c, r) 里的 c ^ (r % 8)
                layout = fx.make_composed_layout(fx.static(fx.SwizzleType.get(3, 3, 3)), layout)
            return fx.make_view(ptr, layout)

        sA, sB = smem(lds.a.ptr, BM), smem(lds.b.ptr, BN)

        # global -> 寄存器 -> LDS：每行 8 个线程 x 16 B，一次 32 行；v1：st_c, st_r = tid % 8, tid // 8
        cp = fx.make_copy_atom(fx.UniversalCopy128b(), fx.BFloat16)
        g2s = fx.make_tiled_copy_tv(cp, fx.make_layout((32, 8), (8, 1)), fx.make_layout((1, 8), (1, 1))).get_slice(tid)
        tAg, tAs = g2s.partition_S(gA), g2s.partition_D(sA)
        tBg, tBs = g2s.partition_S(gB), g2s.partition_D(sB)
        rA, rB = fx.make_fragment_like(tAs), fx.make_fragment_like(tBs)

        # 4 个 wave 排成 2x2；v1：wm, wn = wave % 2, wave // 2
        atom = fx.make_mma_atom(fx.rocdl.MFMA(16, 16, 16, fx.BFloat16))
        tiled_mma = fx.make_tiled_mma(atom, fx.make_layout((2, 2, 1), (1, 2, 0)),
                                      fx.make_tile(None, None, fx.make_layout((4, 4, 2), (1, 8, 4))))
        thr_mma = tiled_mma.thr_slice(tid)
        s2rA = fx.make_tiled_copy_A(cp, tiled_mma).get_slice(tid)
        s2rB = fx.make_tiled_copy_B(cp, tiled_mma).get_slice(tid)
        tAs2r, tBs2r = s2rA.partition_S(sA), s2rB.partition_S(sB)
        fA, fB, fC = thr_mma.make_fragment_A(sA), thr_mma.make_fragment_B(sB), thr_mma.make_fragment_C(gC)
        fAr, fBr = s2rA.retile(fA), s2rB.retile(fB)
        fC.fill(0)

        def load_global(kt):
            fx.copy(cp, tAg[None, None, None, kt], rA)
            fx.copy(cp, tBg[None, None, None, kt], rB)

        if fx.const_expr(prefetch):
            load_global(0)
        for kt in range(fx.Int32(0), fx.Int32(K // BK), fx.Int32(1)):
            if fx.const_expr(not prefetch):
                load_global(kt)
            gpu.barrier()                           # 上一轮的 LDS 读完了
            fx.copy(cp, rA, tAs)
            fx.copy(cp, rB, tBs)
            gpu.barrier()                           # 这一轮的 LDS 写完了
            if fx.const_expr(prefetch):             # 寄存器 rA/rB 已空出来：发下一轮的 global 读
                nxt = fx.Int32(kt) + 1
                load_global(nxt - fx.Int32(nxt == fx.Int32(K // BK)))   # 最后一轮重复读最后一块，省掉分支
            if fx.const_expr(split):
                for s in fx.range_constexpr(BK // 32):
                    fx.copy(cp, tAs2r[None, None, s], fAr[None, None, s])
                    fx.copy(cp, tBs2r[None, None, s], fBr[None, None, s])
                    fx.gemm(atom, fC, fA[None, None, (None, s)], fB[None, None, (None, s)], fC, traversal_order=order)
            else:
                fx.copy(cp, tAs2r, fAr)
                fx.copy(cp, tBs2r, fBr)
                fx.gemm(atom, fC, fA, fB, fC, traversal_order=order)
        cpC = fx.make_copy_atom(fx.UniversalCopy32b(), fx.Float32)
        r2g = fx.make_tiled_copy_C(cpC, tiled_mma).get_slice(tid)
        fx.copy(cpC, r2g.retile(fC), r2g.partition_S(gC))

    @flyc.jit
    def launch(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor, stream: fx.Stream):
        gemm_tiled_lds(A, B, C).launch(grid=(M // BM, N // BN, 1), block=(256, 1, 1), stream=stream)

    return launch


print("代码行数：build_v1", lines(build_v1), "| build_v2", lines(build_v2), "| build_tiled_lds（含两个开关）", lines(build_tiled_lds))

代码行数：build_v1 79 | build_v2 161 | build_tiled_lds（含两个开关） 66


在 4096x4096x8192 上和 v1（两种 swizzle 写法）、v2 比较。bank conflict 用 rocprofv3 的 `SQ_LDS_BANK_CONFLICT` 在 512x512x512 上测（另起一个进程）：

In [17]:
ROCPROF = shutil.which("rocprofv3") or "/opt/rocm/bin/rocprofv3"


def bank_conflicts(cases):
    """cases: {名字: 构造 launcher 的表达式字符串}，在另一个进程里用 rocprofv3 跑"""
    src = "\n".join([
        "import sys, torch",
        f"sys.path.insert(0, {os.path.abspath('../kernels/gemm')!r})",
        "import flydsl.compiler as flyc, flydsl.expr as fx",
        "from flydsl.expr import gpu",
        "from v1 import build_v1",
        "from v2 import build_v2",
        inspect.getsource(build_tiled_lds),
        "M = N = K = 512",
        "A = torch.randn(M, K, dtype=torch.bfloat16, device='cuda')",
        "B = torch.randn(N, K, dtype=torch.bfloat16, device='cuda')",
        "C = torch.zeros(M, N, device='cuda')",
        "s = fx.Stream(torch.cuda.current_stream().cuda_stream)",
        *[f"{expr}(A, B, C, s); torch.cuda.synchronize()" for expr in cases.values()],
    ])
    with tempfile.TemporaryDirectory() as d:
        with open(os.path.join(d, "run.py"), "w") as f:
            f.write(src)
        subprocess.run([ROCPROF, "--pmc", "SQ_LDS_BANK_CONFLICT", "--kernel-include-regex", "^gemm_",
                        "-d", d, "-o", "pmc", "-f", "csv", "--", sys.executable, "run.py"],
                       check=True, capture_output=True, cwd=d)
        path, = glob.glob(f"{d}/**/*counter_collection.csv", recursive=True)
        with open(path) as f:
            values = [int(float(r["Counter_Value"])) for r in sorted(csv.DictReader(f), key=lambda r: int(r["Dispatch_Id"]))]
    return dict(zip(cases, values))


M, N, K = 4096, 4096, 8192
A = torch.randn(M, K, dtype=torch.bfloat16, device="cuda") / 90
B = torch.randn(N, K, dtype=torch.bfloat16, device="cuda") / 90
C = torch.zeros(M, N, device="cuda")
ref = A.float() @ B.float().T
cases = {
    "v1 手写 swizzle": "build_v1(256, 256, 1, M, N, K)",
    "v1 fly swizzle": "build_v1(256, 256, 1, M, N, K, SWIZZLE='fly')",
    "tiled": "build_tiled_lds(256, 256, M, N, K)",
    "tiled 默认顺序": "build_tiled_lds(256, 256, M, N, K, order=None)",
    "tiled 不 swizzle": "build_tiled_lds(256, 256, M, N, K, swizzle=False)",
    "tiled prefetch": "build_tiled_lds(256, 256, M, N, K, prefetch=True)",
    "tiled prefetch+split": "build_tiled_lds(256, 256, M, N, K, prefetch=True, split=True)",
    "v2": "build_v2(256, 256, 1, M, N, K)",
}
conflicts = bank_conflicts(cases)
rows = {}
for name, expr in cases.items():
    build = lambda: eval(expr)
    s = isa_stats(build, A, B, C, stream)
    rows[name] = {**measure(build, A, B, C, ref), "bank冲突": conflicts[name],
                  "VGPR 942/950": f"{s['gfx942']['VGPR']}/{s['gfx950']['VGPR']}",
                  "循环指令": s["gfx942"]["循环指令"], "accvgpr 搬运": s["gfx942"]["accvgpr 搬运"]}
table(rows)
# → v1 两种约 540～550；tiled 约 525～535；不 swizzle 约 500，有 bank conflict；prefetch 约 585，超过 v1；v2 约 670 TFLOPS
# → 除不 swizzle 的 tiled 外 bank conflict 都是 0

                     | 正确 | 编译 ms |    us | TFLOPS | bank冲突 | VGPR 942/950 | 循环指令 | accvgpr 搬运
v1 手写 swizzle      | True |    1102 | 497.7 |  552.3 |        0 |      436/432 |      545 |          107
v1 fly swizzle       | True |    1114 | 511.3 |  537.6 |        0 |      464/464 |      569 |          135
tiled                | True |     751 | 525.6 |  523.0 |        0 |      512/512 |      659 |          193
tiled 默认顺序       | True |     732 | 514.9 |  533.9 |        0 |      512/512 |      646 |          189
tiled 不 swizzle     | True |     696 | 550.1 |  499.6 |    49152 |      510/510 |      650 |          182
tiled prefetch       | True |     729 | 471.5 |  582.9 |        0 |      477/484 |      418 |           18
tiled prefetch+split | True |     722 | 470.1 |  584.7 |        0 |      422/424 |      422 |           26
v2                   | True |    1506 | 406.2 |  676.7 |        0 |      512/512 |      402 |            5


- **正确性和 LDS 访问**：都正确；tiled 版用 composed layout 做 swizzle，bank conflict 和 v1 一样是 0，去掉 swizzle 就有冲突、慢约 5%。LDS 指令条数相同（每轮 16 条 `ds_write_b128` + 32 条 `ds_read_b128`）。
- **寄存器**：直接对应 v1 的 tiled 版把寄存器用满（512，含 AGPR），比 v1 多约 75 个，多出来的是 `v_accvgpr_*` 搬运；这里慢 3%～5%。
- **改结构很便宜**：`prefetch` 只是把两行 `fx.copy` 挪了位置，就从 v1 水平提到约 585 TFLOPS，寄存器和 `accvgpr` 搬运也降下来。手写版要做同样的事，得把 staging 寄存器变成 loop-carried 值，改 `init` / `yield` 和循环前后的代码。`split` 说明切好的 tensor 还可以再按下标切，粒度可以细到单条 copy / 单个 MFMA 块（这里对性能没有帮助，但寄存器又少了约 55 个）。
- **v2 仍然领先约 15%**：它的收益来自精细的软件流水——tile 拆成 4 个象限，每个象限的 64 条 MFMA 中间穿插下一象限的 LDS 读、下一轮的 LDS 写和再下一轮的 global 读，用 `sched_group_barrier` 指定交错比例，只用一份 LDS、两个 barrier。这些决定和"元素在哪个 lane"无关，tiled 写法同样可以做（把 partition 后的 tensor 按象限切开，`rocdl.sched_*` 对任何代码都起作用），但本实验没有实现，需要和 v2 一样的设计工作量。

## 7. 各维度汇总与建议

| 维度 | 手算偏移（v0/v1/v2） | tiled_copy / tiled_mma |
|---|---|---|
| 排布从哪来 | 自己从 MFMA 文档推导，A/B/C、global/LDS 各写一遍 | atom 自带，copy 从 mma 推出；可打印 TV layout 检查（1、2.1 节） |
| 写错的后果 | 下标错往往只是结果不对，没有任何报错 | partition / retile 的 shape 由 layout 推导，三处排布天然一致 |
| 生成的 ISA | 指令选择相同；从大 view 直接取下标时每个 load 单独算地址 | 指令相同，地址是"指针 + 常数"；手写版先切动态坐标也能做到（3、4.1 节） |
| 编译时间 | Python 列表展开，较慢 | 这里快 1.5～2.4 倍（3、6 节） |
| 性能 | 写什么就是什么 | `fx.gemm` 的 `traversal_order` 要和 load 顺序匹配，默认值在这里慢 25%（3、4.2 节） |
| 换 MFMA / 架构 | 重写所有排布相关代码 | 改 atom 参数；指令是否被目标架构支持仍要自己保证（5 节） |
| LDS + swizzle | 手写 XOR，或 composed layout 只算地址再建普通视图 | composed layout 的 LDS tensor 可以直接 `fx.copy`（6 节） |
| 流水 / 调度 | 完全可控，但每改一次结构都要改 loop-carried 状态 | 挪 `fx.copy` 即可改流水；细粒度靠切 tensor + `rocdl.sched_*` |
| 寄存器 | 粒度细，容易控制 | 片段按整个 tile 分配，可能多用寄存器；用切片缩小（6 节 `split`） |

建议：

1. **新写一个 MFMA kernel，默认用 tiled**：先用 `tiled_mma` + `make_tiled_copy_A/B/C` 写出正确版本，排布不用自己推，换 dtype / MFMA 指令 / 架构的成本最低。`traversal_order` 要和 copy 的顺序搭配（4.2 节）。
2. **性能调优阶段仍然用 tiled 做"坐标"，手动做"调度"**：prefetch、多级缓冲、按象限切分，都可以在切好的 tensor 上用下标和 `fx.copy` / `fx.gemm` 的位置表达，再配合 `rocdl.sched_*`。这一阶段的工作量主要在流水设计，和用哪种写法关系不大。
3. **手算偏移适合这些情况**：tiled 没有覆盖的数据排布（例如 [v0_preshuffle](../kernels/gemm/v0_preshuffle.py) 这类为某条指令重排过的输入、非常规的 scale / 量化格式）；需要逐条控制寄存器（v2 那样把操作数当 loop-carried 值精确管理）；或者调试时想看清每一条 load 的来源。
4. 两种写法可以混用：同一个 kernel 里用 tiled 生成 MFMA 片段、手写 global 地址（或反过来）都可以，关键是用 2.1 节的方法确认两边的排布一致。手写时记住 4.1 节的规则：动态坐标先切，静态下标后取。

## 8. 讨论：tiled 是不是一层"枷锁"

一个常见的感受是：tiled 这套 API 虽然容易修改，但晦涩、不直观、学习曲线陡。MFMA 的排布、外存数据的排布这些对优化至关重要的东西被封装以后，变得不直观、不灵活，像套上了一层枷锁。结合本实验的结果看，这不完全是偏见，但也有一部分难度是两种写法都绕不开的。

**有实据的部分**

- **对性能关键的决定被藏起来了**：`fx.gemm` 的默认遍历顺序比 MNK 慢 25%（3 节），签名里看不出来；顺序名字的含义要从 LLVM IR 反推（4.2 节）；真正决定快慢的"load 发出顺序和 MFMA 消费顺序是否匹配"分散在 `fx.copy` 的先后和 `traversal_order` 两处，哪一处都没有直接写着。
- **记号难读**：`((16,4),(4,(1,2))):((1,128),(16,(0,64)))` 信息完整，但要按 1.5 节的方法展开才看得懂；手写的 `row = lane % 16` 一眼就懂。
- **资源不透明**：fragment 按整个 tile 分配，第 6 节的 tiled 版把寄存器用满（512），只能事后看 ISA 才知道；手写版每个 `Vector` 占多少寄存器写的时候就清楚。
- **错误信息离原因远**：在运行时循环里调用 `retile`，报的是 `ThrCopy.__init__` 的 `TypeError`，和真实原因（循环状态里带上了 `ThrCopy` 对象）隔了好几层。

**不完全是 tiled 的问题的部分**

- **MFMA 排布本身就难**，两种写法都要懂。tiled 把它变成一个可以打印、可以断言的值；手写版把它散在 `row`、`kg`、`4*kg + r` 这些式子里。初学者觉得陡，很大一部分是 CuTe 的嵌套 `shape:stride` 记号本身的门槛。
- **手写版的"可控"也有一部分是错觉**：v0 快是因为调度器碰巧把 A、B 的 load 交替发出。只做一个看似无害的改动（先切动态坐标、少算地址），它就掉到和 tiled 默认顺序一样慢（4 节）。两种写法真正决定性能的都是最后的调度，写源码时都看不到，手写只是让人觉得看得到。
- **外存的 layout 并没有被封装掉**：`make_view` / `flat_divide` 仍然由自己写，tiled 接管的只是"切给哪个线程"这一步。

**结论**：这是一个真实的取舍。tiled 的设计目标是正确性和可移植性（换 atom、换架构不出错），而优化需要的是可见性（每条 load 在哪发、等什么、占哪些寄存器）。两者冲突时，tiled 把决定做成了隐式的默认值，所以会有"枷锁"的感觉。

比较实用的折中：

- **把 tiled 当计算器和校验工具**：用它算 TV layout，和手写公式做断言（1.5、2.1 节），kernel 主体仍然手写。
- **或者只让它管最难推的一层**：`tiled_mma` 生成 MFMA fragment 的排布，global / LDS 的地址和 load 顺序自己写（遵守 4.1 节的规则）。
- **无论哪种，都把隐藏的决定显式化**：遍历顺序永远显式传，关键 layout 打印出来或画出来（第 1 节的图就是这样生成的），性能结论以 ISA 为准。